In [3]:
!pip -q install transformers

In [5]:
import torch
from transformers import AutoTokenizer, AutoModel

sentences = [
    "Natural language processing is interesting.",
    "Machine learning helps computers learn."
]
device = "cuda" if torch.cuda.is_available() else "cpu"

def get_embeddings(name, sentences):
    tokenizer = AutoTokenizer.from_pretrained(name)
    model = AutoModel.from_pretrained(name).to(device)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer(sentences, return_tensors="pt",
                       padding=True, truncation=True).to(device)

    with torch.no_grad():
        hidden = model(**inputs).last_hidden_state
        mask = inputs["attention_mask"].unsqueeze(-1)
        embeddings = (hidden * mask).sum(dim=1) / mask.sum(dim=1)

    return embeddings

bert = get_embeddings("bert-base-uncased", sentences)
gpt2 = get_embeddings("gpt2", sentences)

print("BERT embeddings:", bert.shape)
print("GPT-2 embeddings:", gpt2.shape)
print("First BERT vector:", bert[0][:10])
print("First GPT-2 vector:", gpt2[0][:10])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

BERT embeddings: torch.Size([2, 768])
GPT-2 embeddings: torch.Size([2, 768])
First BERT vector: tensor([ 0.2468,  0.0484, -0.3383, -0.1548,  0.0615, -0.3316, -0.0126,  0.6003,
        -0.0525, -0.1626])
First GPT-2 vector: tensor([-0.2379, -0.0615, -0.9781, -0.2360, -0.1374,  0.1926,  6.1881,  0.6679,
        -0.4163, -0.1837])
